In [40]:
input_file = '/Users/liuda/Local/data/trading/data/output/trigger/AMZN.csv'


import pandas as pd
ticker_data = pd.read_csv(input_file)
ticker_data.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Adj Open,Adj High,Adj Low,...,T_T_ShortBullBuy1,T_T_ShortBullBuy2,T_T_LongBullBuy1_0,T_T_LongBullBuy1_1,T_T_LongBullBuy1_2,T_T_LongBullBuy1_3,T_T_LongBullBuy2_0,T_T_LongBullBuy2_1,T_T_LongBullBuy2_2,T_T_LongBullBuy2_3
0,2025-01-02,222.03,225.15,218.19,220.22,220.22,33956600.0,222.03,225.15,218.19,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-03,222.51,225.36,221.62,224.19,224.19,27515606.0,222.51,225.36,221.62,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-04,222.51,225.36,221.62,224.19,224.19,27515606.0,222.51,225.36,221.62,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-05,222.51,225.36,221.62,224.19,224.19,27515606.0,222.51,225.36,221.62,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-06,226.78,228.84,224.84,227.61,227.61,31849831.0,226.78,228.84,224.84,...,0,0,0,0,0,0,0,0,0,0


In [43]:
# key_event_sma_5_cross_sma_20_and_retreat
ticker_data = ticker_data.assign(
	key_event_technical_trend_change=None,
	key_event_institution_op_ratio=0.0,
	key_event_sma_20_retreat=0,
	key_event_sma_5_cross_sma_20_and_retreat=0
)


N = 20
cond1 = (ticker_data['SMA_5_day_Cross_SMA_20_day'] == 1)                  # 当天是上叉
cond2 = ((ticker_data[f'SMA_5_day_Cross_SMA_20_day'] > 0).astype(
int).rolling(window=N, min_periods=1).sum() == 1)  # 近20天内只有一个上叉
cond3 = ((ticker_data[f'SMA_5_day_Cross_SMA_20_day'] < 0).astype(
int).rolling(window=N, min_periods=1).sum() > 0)  # 近20天内有过下叉

ticker_data.loc[cond1 & cond2 & cond3, 'key_event_sma_5_cross_sma_20_and_retreat'] = -1  # 危险信号

cond1 = (ticker_data['SMA_5_day_Cross_SMA_20_day'] == -1)                 # 当天是下叉
cond2 = ((ticker_data[f'SMA_5_day_Cross_SMA_20_day'] < 0).astype(
int).rolling(window=N, min_periods=1).sum() == 1)  # 近20天内只有一个下叉
cond3 = ((ticker_data[f'SMA_5_day_Cross_SMA_20_day'] > 0).astype(
int).rolling(window=N, min_periods=1).sum() > 0)  # 近20天内有过上叉

ticker_data.loc[cond1 & cond2 & cond3, 'key_event_sma_5_cross_sma_20_and_retreat'] = 1    # 买入信号

In [45]:
ticker_data[ticker_data['key_event_sma_5_cross_sma_20_and_retreat'] != 0][['Date', 'SMA_5_day_Cross_SMA_20_day', 'key_event_sma_5_cross_sma_20_and_retreat']]

,Date,SMA_5_day_Cross_SMA_20_day,key_event_sma_5_cross_sma_20_and_retreat
89,2025-04-01,-1.0,1


In [32]:
col_A = 'SMA_5_day'
col_B = 'SMA_20_day'
N = 50

ticker_data[f'{col_A}_Cross_{col_B}_cumsum_{N}'] = (
	(ticker_data[f'{col_A}_Cross_{col_B}'] > 0)
	.astype(int)
	.rolling(window=N, min_periods=1)
	.sum())

# 下叉
ticker_data[f'{col_A}_down_Cross_{col_B}_cumsum_{N}'] = (
	(ticker_data[f'{col_A}_Cross_{col_B}'] < 0)
	.astype(int)
	.rolling(window=N, min_periods=1)
	.sum())


In [33]:
ticker_data = ticker_data.assign(
	key_event_sma_5_cross_sma_20_and_retreat=0,
	key_event_sma_20_retreat=0,
	key_event_sma_5_cross_sma_20_and_retreat_test=0,
)

In [35]:
cond1 = (
    (ticker_data['SMA_5_day_Cross_SMA_20_day'] == 1)                  # 当天是上叉
    & (ticker_data['SMA_5_day_Cross_SMA_20_day_cumsum_50'] == 1)      # 近20天内只是一个上叉
    & (ticker_data['SMA_5_day_down_Cross_SMA_20_day_cumsum_50'] > 0)  # 近20天内有过下叉
)

ticker_data.loc[cond1, 'key_event_sma_5_cross_sma_20_and_retreat'] = -1  # 危险信号

N = 50
cond1 = (ticker_data['SMA_5_day_Cross_SMA_20_day'] == 1)                  # 当天是上叉
cond2 = ((ticker_data[f'SMA_5_day_Cross_SMA_20_day'] > 0).astype(
    int).rolling(window=N, min_periods=1).sum() == 1)  # 近20天内只是一个上叉
cond3 = ((ticker_data[f'SMA_5_day_Cross_SMA_20_day'] < 0).astype(
    int).rolling(window=N, min_periods=1).sum() > 0)  # 近20天内有过下叉


ticker_data.loc[cond1 & cond2 & cond3, 'key_event_sma_5_cross_sma_20_and_retreat_test'] = -1  # 危险信号

In [37]:
pd.set_option('display.max_rows', None)

In [38]:
ticker_data[['Date', "SMA_5_day_Cross_SMA_20_day", 'key_event_sma_5_cross_sma_20_and_retreat', 'key_event_sma_5_cross_sma_20_and_retreat_test']]

,Date,SMA_5_day_Cross_SMA_20_day,key_event_sma_5_cross_sma_20_and_retreat,key_event_sma_5_cross_sma_20_and_retreat_test
0,2025-01-02,0.0,0,0
1,2025-01-03,0.0,0,0
2,2025-01-04,0.0,0,0
3,2025-01-05,0.0,0,0
4,2025-01-06,0.0,0,0
5,2025-01-07,0.0,0,0
6,2025-01-08,0.0,0,0
7,2025-01-09,0.0,0,0
8,2025-01-10,0.0,0,0
9,2025-01-11,0.0,0,0


In [39]:

ticker_data[ticker_data['key_event_sma_5_cross_sma_20_and_retreat'] != 0][
	['Date', 'key_event_sma_5_cross_sma_20_and_retreat', 'key_event_sma_5_cross_sma_20_and_retreat_test']]

,Date,key_event_sma_5_cross_sma_20_and_retreat,key_event_sma_5_cross_sma_20_and_retreat_test
84,2025-03-27,-1,-1
